In [2]:
## MHA

import torch

from torch import nn

class MHA(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        assert self.hidden_dim % self.num_heads == 0
        self.head_dim = hidden_dim // num_heads

        self.linear_q = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.linear_k = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.linear_v = nn.Linear(self.hidden_dim, self.hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.output_projection = nn.Linear(self.hidden_dim, self.hidden_dim)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape
        q = self.linear_q(x)
        k = self.linear_k(x)
        v = self.linear_v(x)

        q = q.view(batch_size, seq_len, self.num_heads, -1)
        k = k.view(batch_size, seq_len, self.num_heads, -1)
        v = v.view(batch_size, seq_len, self.num_heads, -1)

        scores = torch.einsum("bshd,bthd->bsht", q, k).transpose(1, 2)

        if attention_mask is not None:
            scores = scores.masked_fill(attention_mask[:, None, None, :] == 0, float("-inf"))
        scores *= (self.head_dim ** (-0.5))
        scores = torch.softmax(scores, dim=-1)
        scores = self.dropout(scores)

        value = torch.einsum("bhst,bthd->bhsd", scores, v).transpose(1, 2).flatten(2)

        return self.output_projection(value)

In [ ]:
## LoRA
import copy
import torch.nn.functional as F

class LoraLinear(nn.Module):
    def __init__(
        self,
        base_layer: nn.Linear,
        r: int = 8,
        alpha: int = 16,
        dropout_p: float = 0.0,
        test_mode: bool = False
    ):
        super().__init__()
        self.base_layer = copy.deepcopy(base_layer)
        self.r = r
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout_p)

        self.lora_A = nn.Parameter(torch.empty((r, base_layer.in_features), dtype=base_layer.weight.dtype))
        self.lora_B = nn.Parameter(torch.empty((base_layer.out_features), r, dtype=base_layer.weight.dtype))
        nn.init.normal_(self.lora_A, mean=0.0, std=0.02)
        if test_mode:
            nn.init.normal_(self.lora_B, mean=0.0, std=0.02)
        else:
            nn.init.zeros_(self.lora_B)
        
        for param in self.base_layer.parameters():
            param.requires_grad = False

    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scaling = float(self.alpha) / float(self.r)
        lora_adjustment = F.linear(F.linear(self.dropout(x), self.lora_A), self.lora_B)
        return self.base_layer(x) + lora_adjustment * scaling


def replace_linear_with_lora(
        module: nn.Module,
        r: int = 8,
        alpha: int = 16,
        dropout_p: float = 0.0,
        embed_requires_grad: bool = False,
        norm_requires_grad: bool = False,
        head_requires_grad: bool = False,
        test_mode: bool = False,
):
    for name, child in module.named_children():
        if any(s in name for s in ["embed", "norm", "head"]):
            for param in child.parameters():
                param.requires_grad = embed_requires_grad if "embed" in name else norm_requires_grad if "norm" in name else head_requires_grad
        elif isinstance(child, nn.Linear):
            setattr(module, name, LoraLinear(child, r=r, alpha=alpha, dropout_p=dropout_p, test_mode=test_mode))
        else:
            replace_linear_with_lora(
                child,
                r, alpha, dropout_p, embed_requires_grad, norm_requires_grad, head_requires_grad, test_mode
            )